# CGLMP Inequalities

Juan Manuel Segura Guatibonza

In [27]:
include("SDP_QI.jl")
using CairoMakie
using LaTeXStrings

\begin{align*}
    \hat{X}_d &= \sum_{j=0}^{d-1} | j \rangle \langle j \oplus 1| \\
    \hat{F}_d &= \frac{1}{\sqrt{d}} \sum_{j,k} \omega_d^{jk} |j \rangle \langle k | \\
    \omega_d &= e^{2\pi i /d}
\end{align*}

The CGLMP ineuality is the following:

\begin{align*}
    I_d = \sum_{k=0}^{d/2 - 1} \left( 1 - \frac{2k}{d-1} \right) &\left[ (P(A_1 = B_1 + k) + P(B_1=A_2 + k +1) + P(A_2 = B_2+k) + (B_2 = A_1+k)) \right. \\
    & \left. - (P(A_1=B_1-k-1) + P(B_1=A_2-k) + P(A_2=B_2-k-1) + P(B_2=A_1-k-1)) \right]
\end{align*}

where

\begin{equation*}
    P(A_a = B_b + k) = \sum_{j=0}^{d-1} P(A_a=j, B_b= j+k \operatorname{mod} d)
\end{equation*}

Los elementos de medición de $A_1$ y $B_1$ son:
\begin{equation*}
    \Pi_{j|1} = |j\rangle \langle j | \quad , \quad j=0,\dots,d-1
\end{equation*}

Los elementos de medición de $A_2$ y $B_2$ son:
\begin{equation*}
    \Pi_{j|2} = \hat{M}_d|j\rangle \langle j | \hat{M}_d^\dagger \quad , \quad j=0,\dots,d-1
\end{equation*}

donde

\begin{equation*}
    \hat{M}_d = \hat{F}_d^{(1-\alpha)} \hat{X}_d^{-\alpha/2} \quad , \quad \alpha \in \left[0, 1 \right]
\end{equation*}

In [3]:
Threads.nthreads()

4

In [28]:
function Optimal_CGLMP(d::Int)
    Alphas = LinRange(0, 1, 50)
    n = length(Alphas)
    Id_array = zeros(Float64, n)
    Entanglement = zeros(Float64, n)
    M_Incomp = zeros(Float64, n)
    Noise = zeros(Float64, n)
    NoiseDual = zeros(Float64, n)

    Threads.@threads for i in 1:n
        α = Alphas[i]
        M = tunning_M(d, α)

        Proj_A1 = [begin 
            A1j = zeros(ComplexF64, d, d)
            A1j[j, j] = 1.0
            A1j  
        end for j in 1:d]

        Proj_A2 = [begin 
            ket = zeros(ComplexF64, d)
            ket[j] = 1.0
            ψ = M * ket
            ψ = ψ / norm(ψ)
            ψ * ψ'
        end for j in 1:d]

        Proj_A = [Proj_A1, Proj_A2]
        Proj_B = [Proj_A1, Proj_A2]

        Id_max, ρ = CGLMP_opt(Proj_A, Proj_B, d)

        Id_array[i] = Id_max
        r = RRE_PPT(ρ, d, d)
        Entanglement[i] = r / (1 + r)
        r = RRMI(Proj_A)
        M_Incomp[i] = r / (1 + r)
        noise = NoiseRobustness(Proj_A)
        Noise[i] = noise
        noise = NoiseRobustness_dual(Proj_A)
        NoiseDual[i] = noise

        println("Terminated process for α= $(α)")
    end

    CairoMakie.activate!(type = "pdf")

    set_theme!(Theme(
        fontsize = 18,
        fonts = (; regular = "CMU Serif"),
        Axis = (
            xgridvisible = true,
            ygridvisible = true,
            topspinevisible = true,
            rightspinevisible = true,
            spinewidth = 1.2,
        )
    ))

    f = Figure(size = (800, 1500))

    ax1 = Axis(f[1, 1],
        ylabel = "CGLMP",
        title = "Optimal violation of the CGLMP inequality (d=$(d))",
        xticks = 0:0.2:1
    )

    ax2 = Axis(f[2, 1],
        ylabel = "Entanglement",
        xticks = 0:0.2:1
    )

    ax3 = Axis(f[3, 1],
        xlabel = L"\alpha",
        ylabel = "Noise Robustness",
        xticks = 0:0.2:1
    )

    ax4 = Axis(f[4, 1],
        xlabel = L"\alpha",
        ylabel = "Noise Robustness Dual",
        xticks = 0:0.2:1
    )

    ax5 = Axis(f[5, 1],
        xlabel = L"\alpha",
        ylabel = "Incompatibility",
        xticks = 0:0.2:1
    )

    linkxaxes!(ax1, ax2, ax3, ax4, ax5)

    lines!(ax1, Alphas, Id_array, linewidth = 2)
    scatter!(ax1, Alphas, Id_array, markersize = 6)

    lines!(ax2, Alphas, Entanglement, linewidth = 2)
    scatter!(ax2, Alphas, Entanglement, markersize = 6)

    lines!(ax3, Alphas, Noise, linewidth = 2)
    scatter!(ax3, Alphas, Noise, markersize = 6)

    lines!(ax4, Alphas, NoiseDual, linewidth = 2)
    scatter!(ax4, Alphas, NoiseDual, markersize = 6)

    lines!(ax5, Alphas, M_Incomp, linewidth = 2)
    scatter!(ax5, Alphas, M_Incomp, markersize = 6)

    hidexdecorations!(ax1, ticks=false, grid=false)
    hidexdecorations!(ax2, ticks=false, grid=false)
    hidexdecorations!(ax3, ticks=false, grid=false)
    hidexdecorations!(ax4, ticks=false, grid=false)

    xlims!(ax5, 0, 1)

    rowgap!(f.layout, 10)

    save("resultsCGLMP/cglmp_d$(d).pdf", f)

    return f
end

Optimal_CGLMP (generic function with 1 method)

In [39]:
CGLMP_analysis(9)
CGLMP_analysis(10)

Terminated process for α = 0.7576
Terminated process for α = 0.5051
Terminated process for α = 0.2525
Terminated process for α = 0.0
Terminated process for α = 0.5152
Terminated process for α = 0.7677
Terminated process for α = 0.0101
Terminated process for α = 0.2626
Terminated process for α = 0.5253
Terminated process for α = 0.0202
Terminated process for α = 0.2727
Terminated process for α = 0.7778
Terminated process for α = 0.5354
Terminated process for α = 0.0303
Terminated process for α = 0.2828
Terminated process for α = 0.7879
Terminated process for α = 0.0404
Terminated process for α = 0.5455
Terminated process for α = 0.798
Terminated process for α = 0.2929
Terminated process for α = 0.0505
Terminated process for α = 0.5556
Terminated process for α = 0.8081
Terminated process for α = 0.303
Terminated process for α = 0.0606
Terminated process for α = 0.5657
Terminated process for α = 0.8182
Terminated process for α = 0.3131
Terminated process for α = 0.5758
Terminated process 

Row,alpha,CGLMP,Entanglement,NoiseDual
,Float64,Float64,Float64,Float64
1,0.0,2.24199,0.930259,0.62008
2,0.010101,2.24161,0.931016,0.62008
3,0.020202,2.24078,0.932974,0.620023
4,0.030303,2.2398,0.934353,0.619962
5,0.040404,2.23879,0.935805,0.619772
6,0.0505051,2.23772,0.935853,0.619528
7,0.0606061,2.23663,0.934974,0.619172
8,0.0707071,2.23552,0.940035,0.618835
9,0.0808081,2.23445,0.94131,0.618394


In [41]:
d= 10

set_theme!(Theme(
    fontsize = 18,
    fonts = (; regular = "CMU Serif"),
    Axis = (
        xgridvisible = true,
        ygridvisible = true,
        topspinevisible = true,
        rightspinevisible = true,
        spinewidth = 1.2,
    )
))

f = Figure(size = (1000, 1500))

ax1 = Axis(f[1, 1],
    ylabel = "CGLMP",
    title = "Optimal violation of the CGLMP inequality (d=$(d))",
    titlesize = 25,
    xticks = 0:0.2:1,
    ylabelsize = 22
)

ax2 = Axis(f[2, 1],
    ylabel = "Entanglement",
    xticks = 0:0.2:1,
    ylabelsize = 22
)

ax3 = Axis(f[3, 1],
    xlabel = L"\alpha",
    ylabel = "Measurement incompatibility",
    xticks = 0:0.2:1,
    xlabelsize = 23,
    ylabelsize = 22
)

linkxaxes!(ax1, ax2, ax3)

df = CSV.read("resultsCGLMP/cglmp_d$(d).csv", DataFrame)
alphas = df.alpha
cglmp = df.CGLMP
entanglement = df.Entanglement
incompatibility = 1 .- df.NoiseDual

lines!(ax1, alphas, cglmp, linewidth = 2)
scatter!(ax1, alphas, cglmp, markersize = 5)

lines!(ax2, alphas, entanglement, linewidth = 2)
scatter!(ax2, alphas, entanglement, markersize = 5)

lines!(ax3, alphas, incompatibility, linewidth = 2)
scatter!(ax3, alphas, incompatibility, markersize = 5)

hidexdecorations!(ax1, ticks=false, grid=false)
hidexdecorations!(ax2, ticks=false, grid=false)

xlims!(ax3, 0, 1)

rowgap!(f.layout, 10)

save("resultsCGLMP/cglmp_d$(d).pdf", f, pt_per_unit=1)

CairoMakie.Screen{PDF}
